# Bài 12: Máy hiểu chữ
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Kho chữ do nhóm tác giả tự ghép từ các mẫu câu về lớp 11A3.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Kho chữ: ghép câu từ các mẫu

In [ ]:
import random
import numpy as np

random.seed(0)
BAN = ["minh", "lan", "khoa", "an"]
MON = ["toán", "văn", "lý", "hoá", "sử"]
MON_AN = ["bánh_mì", "phở", "xôi", "cơm_tấm"]
XE = ["xe_đạp", "xe_buýt", "xe_máy_điện"]
GV = {"cô_hạnh": "toán", "cô_mai": "văn", "thầy_nam": "lý",
      "cô_thu": "hoá", "thầy_bình": "sử"}

kho = []
for _ in range(400):
    b, m = random.choice(BAN), random.choice(MON)
    gv, mon_gv = random.choice(list(GV.items()))
    kho += [f"{b} thích học {m}", f"{b} làm bài tập {m} ở nhà",
            f"{gv} dạy {mon_gv} ở lớp", f"{gv} chấm bài {mon_gv} rất nhanh",
            f"giờ {mon_gv} của {gv} rất vui", f"{b} hỏi {gv} bài {mon_gv}",
            f"sáng nay {b} ăn {random.choice(MON_AN)} ở căng_tin",
            f"{b} đi học bằng {random.choice(XE)}"]
print("Số câu:", len(kho))
print(kho[:3])

### Ô 2. Đếm xem mỗi từ hay đứng gần những từ nào

In [ ]:
tu_dien = sorted({t for cau in kho for t in cau.split()})
vi_tri = {t: i for i, t in enumerate(tu_dien)}
N = len(tu_dien)
dem = np.zeros((N, N))
for cau in kho:
    ds = cau.split()
    for i, t in enumerate(ds):
        for j in range(max(0, i - 2), min(len(ds), i + 3)):   # cửa sổ 2 từ mỗi bên
            if j != i:
                dem[vi_tri[t], vi_tri[ds[j]]] += 1
print("Số từ khác nhau:", N)
gan = np.argsort(-dem[vi_tri["toán"]])[:5]
print('"toán" hay đứng gần:', [tu_dien[k] for k in gan])

### Ô 3. Nén bảng đếm thành vectơ 10 chiều

In [ ]:
tong = dem.sum()
p_tu = dem.sum(axis=1) / tong
with np.errstate(divide="ignore"):
    pmi = np.log((dem / tong) / np.outer(p_tu, p_tu))
pmi = np.maximum(pmi, 0)                         # chỉ giữ liên hệ dương
U, S, Vt = np.linalg.svd(pmi)
vec = U[:, :10] * S[:10]                         # mỗi từ là một vectơ 10 số
print('Vectơ của "toán":', vec[vi_tri["toán"]].round(2))

### Ô 4. Đo độ giống nhau bằng cosine

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def giong(a, b):
    return cosine(vec[vi_tri[a]], vec[vi_tri[b]])

for a, b in [("toán", "văn"), ("toán", "phở"), ("phở", "xôi"),
             ("xe_đạp", "xe_buýt"), ("minh", "lan"), ("minh", "toán")]:
    print(f"{a:8} - {b:8}: {giong(a, b):5.2f}")

## Chặng 2: Đọc
1. Ở ô 2, máy có biết 'toán' là một môn học không? Nó chỉ biết điều gì?
2. Ở ô 4, cặp nào giống nhau nhất, cặp nào khác nhau nhất? Vì sao?
3. Cosine bằng 1, bằng 0, bằng −1 nghĩa là gì?

## Chặng 3: Sửa
### Ô 5. Tìm những từ gần nhất

In [ ]:
def gan_nhat(v, bo_qua=(), k=3):
    diem = [(cosine(v, vec[i]), t) for t, i in vi_tri.items() if t not in bo_qua]
    return [t for _, t in sorted(diem, reverse=True)[:k]]

for t in ["toán", "phở", "xe_buýt", "cô_hạnh"]:
    print(f"Gần '{t}' nhất:", gan_nhat(vec[vi_tri[t]], bo_qua=[t]))

### Ô 6. Phép tính trên từ: cô_hạnh − toán + văn = ?

In [ ]:
def tuong_tu(a, b, c):
    v = vec[vi_tri[a]] - vec[vi_tri[b]] + vec[vi_tri[c]]
    return gan_nhat(v, bo_qua=[a, b, c], k=1)[0]

thu = [("cô_hạnh", "toán", "văn", "cô_mai"), ("thầy_nam", "lý", "sử", "thầy_bình"),
       ("cô_mai", "văn", "hoá", "cô_thu"), ("toán", "cô_hạnh", "cô_mai", "văn"),
       ("văn", "cô_mai", "thầy_nam", "lý")]
dung = 0
for a, b, c, mong_doi in thu:
    kq = tuong_tu(a, b, c)
    dung += kq == mong_doi
    print(f"{a} - {b} + {c} = {kq:10} (mong đợi: {mong_doi})")
print("Đúng", dung, "trên", len(thu))

### Ô 7. So với cách đếm từ của Bài 3 (Tập 1)
Với cách đếm từ, mỗi từ là một cột riêng. Hai từ khác nhau luôn có cosine bằng 0.

In [ ]:
mot_cot = np.eye(N)            # mỗi từ một cột, như CountVectorizer
a, b = mot_cot[vi_tri["toán"]], mot_cot[vi_tri["văn"]]
print("Đếm từ    - toán và văn:", cosine(a, b))
print("Vectơ từ  - toán và văn:", round(giong("toán", "văn"), 2))

## Mở rộng (kết quả tuỳ máy)
### Ô 8. Tách từ tiếng Việt bằng thư viện
Ở ô 1, sách tự nối 'xe_đạp' bằng dấu gạch dưới. Thư viện `underthesea` làm việc này tự động.

In [ ]:
!pip install -q underthesea
from underthesea import word_tokenize
print(word_tokenize("Sáng nay Minh đi học bằng xe đạp", format="text"))

## Chặng 4: Tự làm
1. Ở ô 3, đổi số chiều từ 10 thành 2 và 20. Kết quả ô 4 và ô 6 thay đổi thế nào?
2. Thêm vào ô 1 một mẫu câu mới, ví dụ `f"{b} uống trà_sữa ở {quán}"`, chạy lại và tìm từ gần 'trà_sữa' nhất.
3. Thử ô 6 với `minh - toán + văn`. Kết quả có ý nghĩa không? Vì sao?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*